In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import torch
from exceptiongroup import catch
from mkl import verbose


class modelAndDataPreparation:
    def __init__(self, moviesFile = "movies.csv", usersFile = "ratings.csv"):
        self.moviesFilePath_ = moviesFile
        self.usersFilePath_ =  usersFile

        self.moviesInfo_ = pd.read_csv('movies.csv')
        self.ratingsInfo_ = pd.read_csv(self.usersFilePath_)
        self.ratingsInfo_ = self.ratingsInfo_.rename(columns={'rating':'weight'})
        self.ratingsInfo_ = self.ratingsInfo_.merge(self.moviesInfo_[['movieId','title']], on='movieId')

        self.movieTraining_ = None
        self.movieTest_ = None

        self.gList_ = None

        self.corrMatrix_ = None
        self.W_ = None
        self.trainAndTestList_ = {}

        self.makeCorrMatrix()
        self.makeGraph()
        self.createTrainAndDataSet()
    def makeCorrMatrix(self):
        movieMatrix = pd.pivot_table(self.ratingsInfo_,values='weight', columns='title', index='userId')

        movieRand = movieMatrix.sample(frac=1, random_state=42)
        n = int(0.8 * len(movieRand))

        movieTraining = movieRand.iloc[:n]
        movieTest = movieRand.iloc[n:]

        self.movieTraining_ = movieTraining
        self.movieTest_ = movieTest

        # Now we compute users correlation
        # We can specify the minimum number of users  who rate the same movie
        # and for that movies we will calculate correlation
        minUsers = 10
        corrMatrix = movieTraining.corr(min_periods=minUsers)
        corrMatrix.fillna(0, inplace=True)

        # 1) retain nMax most correlated neighbors
        # 2) all diagonal elements will be zero
        # 3) normalization by its spectral radius
        nMax = 35
        ranked = corrMatrix.rank(axis=1, method='min', ascending=False)
        mask = ranked>nMax

        self.corrMatrix_ = corrMatrix.mask(mask, 0)

        return True
    def makeGraph(self):
        G = nx.from_pandas_adjacency(self.corrMatrix_, create_using=nx.Graph)

        # Because of our conditions nMax and minUsers there can happen isolated
        # node in graph. We will analise connected components and get the largest
        # for the further analise

        ConnectedComponents = nx.connected_components(G)
        # sort increasingly
        ConnectedComponents = sorted(ConnectedComponents, key = len, reverse=True)
        G = G.subgraph(ConnectedComponents[0])

        self.gList_ = [node for node in self.corrMatrix_.index if node in G.nodes()]

        # remove diagonal elements and set to zero (we want to remove self-loops)
        W = nx.to_numpy_array(G)

        for i in range(W.shape[0]):
            for j in range(W.shape[1]):
                if i == j:
                    W[i][j] = 0

        eigValuesW,_= np.linalg.eig(W)

        idx = np.argsort(np.abs(eigValuesW))[::-1]
        eigValuesW = eigValuesW[idx]

        self.W_ = W/abs(eigValuesW[0])

        W = self.W_
        rows, cols = np.nonzero(W)
        self.edgeIndex_ = torch.tensor([rows, cols], dtype=torch.long)
        self.edgeWeight_ = torch.tensor(W[rows, cols], dtype=torch.float32)
    def prepareData(self, movie,train=True):
        if train == True:
            userMovieScoreMatrix = self.movieTraining_
        else:
            userMovieScoreMatrix = self.movieTest_

        moviesInGraph = self.gList_
        W = self.W_

        userMovieScoreMatrix = userMovieScoreMatrix.dropna(axis=0,subset=[movie])
        userMovieScoreMatrix = userMovieScoreMatrix[moviesInGraph]

        #############
        # format set
        #############
        ratingsMovie = userMovieScoreMatrix[movie]

        userMovieScoreMatrix[movie] = 0
        xData = userMovieScoreMatrix\
        .fillna(0)\
        .to_numpy()\
        .reshape(
            len(userMovieScoreMatrix),
            1,
            W.shape[0]
        )

        userMovieScoreMatrix[:] = 0
        userMovieScoreMatrix[movie] = ratingsMovie

        yData = userMovieScoreMatrix\
        .fillna(0)\
        .to_numpy()\
        .reshape(
            len(userMovieScoreMatrix),
            1,
            W.shape[0]
        )

        return xData, yData
    def createTrainAndDataSet(self):
        k = 0
        for movieName in self.gList_:
            xDataTrain, yDataTrain = self.prepareData(movieName,True)
            xDataTest, yDataTest = self.prepareData(movieName, False)

            self.trainAndTestList_[movieName] = {
            "x_train": xDataTrain,
            "y_train": yDataTrain,
            "x_test": xDataTest,
            "y_test": yDataTest
            }
            k += 1

In [ ]:
from torch_geometric.utils import from_networkx
from torch_geometric.data import Data
import torch
from tqdm import trange
import os
from collections import defaultdict
import heapq

from torch.nn import ReLU, Linear, Dropout
from torch_geometric.nn import Sequential, TAGConv
from torch_geometric.loader import DataLoader

class GNN:
    def __init__(self, Data : modelAndDataPreparation):
        self.Data_ = Data
        df = pd.read_csv("systemTestRating.csv")
        longDf = df.melt(id_vars=["username"], var_name="movie", value_name="rating")
        users = defaultdict(dict)

        for r in longDf.to_dict(orient="records"):
            users[r["username"]][r["movie"]] = r["rating"]
        self.activeUsers_ = dict(users)

        self.alpha_ = 1
        self.K_ = 3


        self.TrainedModels_ = {}
        self.adaptedTrainData_ = {}
        self.adaptedTestData_ = {}

        self.GraphW_ = nx.from_numpy_array(self.Data_.W_)
        self.pygGraph_ = from_networkx(self.GraphW_)

        self.loadModels()
    def adaptData(self):
        for elem in self.Data_.gList_:
            pygTrainData = []
            xDataTrain = self.Data_.trainAndTestList_[elem]['x_train']
            yDataTrain = self.Data_.trainAndTestList_[elem]['y_train']
            for iin, out in zip(xDataTrain, yDataTrain):
                iinTorch = torch.from_numpy(iin).float().T
                outTorch = torch.from_numpy(out).float().T
                trainGraph = Data(edge_index=self.pygGraph_.edge_index,
                            weight=self.pygGraph_.weight,
                            x=iinTorch,
                            y=outTorch
                            )
                pygTrainData.append(trainGraph)

            self.adaptedTrainData_[elem] = pygTrainData

            pygTestData = []
            xDataTest = self.Data_.trainAndTestList_[elem]['x_test']
            yDataTest = self.Data_.trainAndTestList_[elem]['y_test']
            for iin, out in zip(xDataTest, yDataTest):
                iinTorch = torch.from_numpy(iin).float().T
                outTorch = torch.from_numpy(out).float().T
                trainGraph = Data(edge_index=self.pygGraph_.edge_index,
                            weight=self.pygGraph_.weight,
                            x=iinTorch,
                            y=outTorch
                            )
                pygTestData.append(trainGraph)

            self.adaptedTestData_[elem] = pygTestData
    def lossGnn(self,pred, true):
        pred = pred.squeeze()
        true = true.squeeze()
        ixMovie = true.nonzero(as_tuple=False)

        res = torch.mean((pred[ixMovie] - true[ixMovie])**2)
        return res
    def makeModel(self, Fin=1, Fout=64, K=4, DropoutRate = 0.0):
        modelGNN = Sequential(
            'x, edge_index, edge_weight',[
                (TAGConv(in_channels = Fin, out_channels = Fout, K = K, normalize = False),'x, edge_index, edge_weight -> x'),
                ReLU(inplace = True),
                Dropout(p = DropoutRate),
                Linear(Fout, Fin)
            ]
        )
        return modelGNN
    def trainGnnModel(self, model, trainData, testData, batcSize=5, epochs=40, learningRate=0.005):
        optimizer = torch.optim.Adam(model.parameters(), lr=learningRate)

        TrainLoader = DataLoader(dataset=trainData, batch_size=batcSize,shuffle=True)
        TestLoader = DataLoader(dataset=testData, batch_size=len(testData), shuffle=True)

        trainLoss = []
        testLoss = []

        for ep in trange(epochs, desc='GNN training', unit='epoch'):
            model.train()
            for batch in TrainLoader:
                optimizer.zero_grad()
                yHat = model(batch.x, batch.edge_index, batch.weight)
                yBatch = batch.y

                loss = self.lossGnn(yHat, yBatch)
                loss.backward()
                optimizer.step()
                trainLoss.append(loss.item())

            for batch in TestLoader:
                yHatTest = model(batch.x, batch.edge_index, batch.weight)
                yBatchTest = batch.y
                loss = self.lossGnn(yHatTest, yBatchTest).item()
                testLoss.append(loss)

        return (model, trainLoss, testLoss)
    def trainModels(self):
        trainLoss = {}
        testLoss = {}

        start = False

        k = 0
        for movie in self.Data_.gList_:
            if movie == "2001: A Space Odyssey (1968)":
                start = True

            if start:
                model = self.makeModel()
                print(f"{str(k)}. Movie: {movie}")
                trainedModel,trll,tstll = self.trainGnnModel(model, self.adaptedTrainData_[movie], self.adaptedTrainData_[movie])

                trainLoss[movie] = trll
                testLoss[movie] = tstll

                with open("trainHistory.txt", "a") as f:
                    out = f"{k}. Model za film {movie} je istreniran\n"
                    f.write(out)

                print("Model je uspesno istreniran!")

                tmp = movie.replace("\\", "___").replace("/", "__").replace(":","____").replace("?","_____").replace("*","______")
                filePath = os.path.join("trainingParameters", f"{tmp}.pth")
                torch.save(model.state_dict(), filePath)
                print("Parametri treniranja su sacuvani")
                k += 1

        df = pd.DataFrame(list(trainLoss.items()), columns=["title", "trainLoss"])
        df.to_csv("trainLoss.csv", index=False)

        df2 = pd.DataFrame(list(testLoss.items()), columns=["title", "testLoss"])
        df2.to_csv("testLoss.csv", index=False)
    def loadModels(self):
        for movie in self.Data_.gList_:
            model = self.makeModel()

            tmp = movie.replace("\\", "___").replace("/", "__").replace(":","____").replace("?","_____").replace("*","______")
            filePath = os.path.join("trainingParameters", f"{tmp}.pth")

            model.load_state_dict(torch.load(filePath))
            self.TrainedModels_[movie] = model
    def activeSpecificModel(self, movieName, userName, ratingSeed):
        if movieName not in self.Data_.gList_:
            predRating = 0
            return predRating

        if userName == "":
            predRating = 0
            return predRating

        try:
            if self.activeUsers_[userName][movieName] != 0:
                predRating = self.activeUsers_[userName][movieName]
                return predRating
        except:
            pass

        model = self.TrainedModels_[movieName]
        movieId = self.Data_.gList_.index(movieName)

        ###############################

        userX = None

        if ratingSeed == None:
            user_ratings = list(self.activeUsers_[userName].values())
            userX = torch.tensor(user_ratings, dtype=torch.float).view(-1, 1)
        else:
            user_ratings = np.zeros((self.Data_.W_.shape[0],))
            k = 0
            for elem in ratingSeed:
                user_ratings[k] = elem
                k += 1

            userX = torch.tensor(user_ratings, dtype=torch.float).view(-1, 1)
        model.eval()
        pred = model(userX, self.Data_.edgeIndex_, self.Data_.edgeWeight_)

        predRating = pred[movieId].item()/5

        return predRating
    def giveReccomendation(self, resKDict, userName,ratingSeed=None):
        movies = {}
        tmp = self.alpha_
        if all(v == 0 for v in resKDict.values()):
            self.alpha_ = 0
        for movie in resKDict:
            predGNN = self.activeSpecificModel(movie, userName, ratingSeed)
            predEMB = resKDict[movie]
            predRES = (1 - self.alpha_)*predEMB + self.alpha_*predGNN

            movies[movie] = predRES

        topKKeys = heapq.nlargest(self.K_, movies, key=movies.get)

        self.alpha_ = tmp

        return topKKeys

In [ ]:
import tensorflow as tf
import sys
import os

from collections import defaultdict

class EpochProgressBar(tf.keras.callbacks.Callback):
    def __init__(self, total_epochs):
        super().__init__()
        self.total_epochs = total_epochs
    def on_epoch_end(self, epoch, logs=None):
        bar_len = 40
        progress = int((epoch + 1) / self.total_epochs * bar_len)
        bar = '[' + '-'*progress + ' '*(bar_len - progress) + ']'
        sys.stdout.write(f'\rEpochs {bar} {epoch+1}/{self.total_epochs}')
        sys.stdout.flush()

class FCNN:
    def __init__(self, Data : modelAndDataPreparation):
        self.Data_ = Data

        df = pd.read_csv("systemTestRating.csv")
        longDf = df.melt(id_vars=["username"], var_name="movie", value_name="rating")
        users = defaultdict(dict)

        for r in longDf.to_dict(orient="records"):
            users[r["username"]][r["movie"]] = r["rating"]

        self.alpha_ = 1
        self.K_ = 3

        self.activeUsers_ = dict(users)
        self.TrainedModels_ = {}
        self.inputLayer_ = len(Data.gList_)

        self.loadModels()
    def makeModel(self, H = 64):
        model = tf.keras.Sequential()
        model.add(tf.keras.Input(shape=(self.inputLayer_,)))
        model.add(tf.keras.layers.Dense(
        H,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
        ))
        model.add(tf.keras.layers.Dense(1, activation='linear'))

        model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['accuracy']
        )

        return model
    def trainFCNNModel(self, movieName):

        xTrain = self.Data_.trainAndTestList_[movieName]['x_train']
        a = xTrain.shape[0]
        c = xTrain.shape[2]
        xTrain = xTrain.reshape(a,c)
        y = self.Data_.trainAndTestList_[movieName]['y_train']
        yTrain = y[y != 0]

        xTest = self.Data_.trainAndTestList_[movieName]['x_test']
        a1 = xTest.shape[0]
        c1 = xTest.shape[2]
        xTest = xTest.reshape(a1,c1)
        y1 = self.Data_.trainAndTestList_[movieName]['y_test']
        yTest = y1[y1 != 0]

        print(f"Treniranje za film {movieName} je započeto!")
        model = self.makeModel()
        totalEpochs = 40

        if len(xTest) > 0:
            history = model.fit(
            xTrain, yTrain,
            epochs=totalEpochs,
            batch_size=5,
            validation_data=(xTest, yTest),
            verbose=0,
            callbacks=[EpochProgressBar(totalEpochs)]
            )
        else:
            history = model.fit(
            xTrain, yTrain,
            epochs=totalEpochs,
            batch_size=5,
            verbose=0,
            callbacks=[EpochProgressBar(totalEpochs)]
            )


        #plt.figure()
        #plt.plot(history.history['loss'])
        #plt.plot(history.history['val_loss'])
        #plt.savefig("FCNN")
        #plt.show()

        return history, model
    def trainModels(self,movieLim):
        k = 0
        trainLoss = {}
        testLoss = {}
        start = False
        for movie in self.Data_.gList_:
            if movie == movieLim:
                start = True
            if start:
                history, model = self.trainFCNNModel(movie)

                tmp = movie.replace("\\", "___").replace("/", "__").replace(":","____").replace("?","_____").replace("*","______")
                filePath = os.path.join("trainingParametersFCNN", f"{tmp}.keras")

                model.save(filePath)
                with open("trainHistoryFCNN.txt", "a") as f:
                        out = f"{k}. Model za film {movie} je istreniran\n"
                        f.write(out)

                print("\nModel je uspesno istreniran!")
                k += 1
                self.TrainedModels_[movie] = model
                if 'val_loss' in history.history:
                    val_loss = history.history['val_loss']
                else:
                    val_loss = "x"
                trainLoss[movie] = history.history['loss']
                testLoss[movie] = val_loss

        df = pd.DataFrame(list(trainLoss.items()), columns=["title", "trainLoss"])
        df.to_csv("trainLossFCNN.csv", index=False)

        df2 = pd.DataFrame(list(testLoss.items()), columns=["title", "testLoss"])
        df2.to_csv("testLossFCNN.csv", index=False)
    def loadModels(self):
        for movie in self.Data_.gList_:

            tmp = movie.replace("\\", "___").replace("/", "__").replace(":","____").replace("?","_____").replace("*","______")
            filePath = os.path.join("trainingParametersFCNN", f"{tmp}.keras")

            model = tf.keras.models.load_model(filePath)
            self.TrainedModels_[movie] = model
    def activeSpecificModel(self,movieName, userName, ratingSeed):
        if movieName not in self.Data_.gList_:
            predRating = 0
            return predRating
        if userName == "":
            predRating = 0
            return predRating
        try:
            if self.activeUsers_[userName][movieName] != 0:
                predRating = self.activeUsers_[userName][movieName]
                return predRating
        except:
            pass
        model = self.TrainedModels_[movieName]

        user_ratings = None


        if ratingSeed == None:
            user_ratings = list(self.activeUsers_[userName].values())
        else:
            user_ratings = np.zeros((self.Data_.W_.shape[0],))
            k = 0
            for elem in ratingSeed:
                user_ratings[k] = elem
                k += 1

        user_ratings = np.array(user_ratings).reshape(1, -1)
        predRating = model.predict(user_ratings)

        return predRating/5
    def giveRecommendation(self, resKDict, userName,ratingSeed=None):
        movies = {}
        tmp = self.alpha_
        if all(v == 0 for v in resKDict.values()):
            self.alpha_ = 0

        for movie in resKDict:
            predFCNN = self.activeSpecificModel(movie, userName, ratingSeed)
            predEMB = resKDict[movie]
            predRES = (1 - self.alpha_)*predEMB + self.alpha_*predFCNN

            movies[movie] = predRES

        topKKeys = heapq.nlargest(self.K_, movies, key=movies.get)

        self.alpha_ = tmp

        return topKKeys

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import ast
from openai import OpenAI
import json

class SearchAgent:
    def __init__(self):
        self.language_ = "English"

        self.client_ = OpenAI()

        self.systemInstruction_ = (
            f"Ti si asistent sistema za preporuku filmova. Pozdravi korisnika imenom, pitaj ga kako se oseca danas, sta bi voleo da gleda i zatraži detaljan opis filma koji voli. "
            "Naglasi da je kvalitetan i bogat opis ključan za precizno preporučivanje. Navedi primer opisa u dve recenice."
            f"Budi ljubazan i profesionalan. Tvoj odgovor treba da bude izmedju 80 i 150 reči.Komuniciraj na {self.language_}"
        )

        self.embModel_="text-embedding-3-small"

        self.filmBase_ = pd.read_csv('embeddings.csv',sep='|')
        self.filmBase_['embedding'] = self.filmBase_['embedding'].apply(ast.literal_eval)

        self.KNN_ = 20

        filmDescBase = pd.read_csv('moj_fajl.csv', sep='|')

        self.filmDescBase_ = dict(zip(filmDescBase['title'], filmDescBase['textToken']))
    def clasicalInteraction(self,userName):
        messages = [
        {"role": "system", "content": self.systemInstruction_},
        {"role": "user", "content": f"Ime korisnika koji želi preporuku filma je {userName}."},
        {"role": "user", "content": f"Pokreni razgovor. Komuniciraj na {self.language_}"}
    ]
        response = self.client_.chat.completions.create(
            model="gpt-4o-mini", # Ili "gpt-4-turbo"
            messages=messages,
            max_tokens=160,
            temperature=0.7
        )
        print(response.choices[0].message.content)
    def interactiveValidation(self):
        messages = [
            {"role": "system", "content": self.systemInstruction_},
            {"role": "system", "content":
                ("Već si poželio dobrodošlicu korisniku.Sada je tvoj cilj je da dobiješ bogat, "
                "detaljan opis filma od korisnika radi kreiranja preciznih embedding vektora. "
                "Analiziraj sve što je korisnik do sada napisao. Ako je opis i dalje štur, "
                "traži dopunu (npr. žanr, atmosfera, glumci). Budi ljubazan i sažet od 50 do 90 rеči."
                 f"Ukoliko ti korisnik daje podatke na nekom jeziku koji nije srpski, ti ga prevedi na srpski, to čuvaj i  sa tim dalje radi.Komuniciraj na {self.language_}.")
            }
        ]

        rejection = 0
        maxRejection = 1

        while True:
            userInput = input("You: ")
            messages.append({"role": "user", "content": userInput})

            promptChek = (
                f"Analiziraj dosadašnji opis. Da li je dovoljno informativan za kvalitetan embedding? Komuniciraj na {self.language_}."
                "Odgovori isključivo u JSON formatu: {'prolaz': bool, 'sugestija': str}"
            )

            validation = self.client_.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages + [{"role": "system", "content": promptChek}],
                response_format={"type": "json_object"}
            )
            result = json.loads(validation.choices[0].message.content)

            if result['prolaz'] or rejection >= maxRejection:
                if rejection >= maxRejection and not result['prolaz']:
                    print("\n It is enough description.")
                else:
                    print("\nDescription is awesome! You will enjoy movie!")

                finalText = " ".join([m['content'] for m in messages if m['role'] == 'user' and m['content'] != "Pozdravi me i zatraži opis."])

                return finalText, self.getEmbedding(finalText)

            else:
                rejection += 1
                print(f"{result['sugestija']} (Improve chances: {maxRejection - rejection + 1})\n")
                messages.append({"role": "assistant", "content": result['sugestija']})
    def getEmbedding(self, text):
        response = self.client_.embeddings.create(
            model=self.embModel_,
            input=text
        )
        return response.data[0].embedding
    def knnEmbd(self,inEmb):

        baseMatrix = np.vstack(self.filmBase_['embedding'].values)
        similarity = cosine_similarity(np.array(inEmb).reshape(1, -1), baseMatrix)[0]
        self.filmBase_['similarity_score'] = similarity
        topK = self.filmBase_.sort_values(by='similarity_score', ascending=False).head(self.KNN_)

        topKDict = topK.set_index('title')['similarity_score'].to_dict()
        return topKDict
    def finalAnswer(self,userName, movieTitles):
        messages = [
        {"role": "system", "content": self.systemInstruction_},
        {"role": "system", "content": f"Korisnik se zove {userName}"},
        {"role": "assistant", "content": "Izvrsili smo pretragu filmova. Nemoj da pozdravljas korisnika. Samo ukratko reci da su predlozi spremni. Naredni tvoj zadatak je da koriscnicima ukratko plasiras filmove koje smo odabradili, a koje cu ti ja dalje reci. Dobices sva imena filmova i njihove opise iz baze, koje treba da mu iskomuniciras."},
        {"role": "user", "content": f"Pokreni razgovor, kad dobijes poruku sa prvim filmom. Za svaki od filmova u petlji koje ti dam daćeš kratak i jasan opis, kao što ću ti sugerisati kroz dalje poruke.Komuniciraj na {self.language_}"}
        ]
        movieNum = 0
        for movie in movieTitles:
            desc = self.filmDescBase_[movie]

            movieMessage = {
            "role": "assistant",
            "content": f"Krecemo sa glavnim zadatkom. Za dati film i njegov opis plasiraj korsiniku predlog.Probaj da imas u vidu filmove koje si prethodno preporucio pa da ne bude da svaki put dajes opis iz prve, pa nekako prirodno nadovezuj, to mozes proveriti kroz prethodne poruke. Hocu da se nadovezujes na prethodne preporuke i da preporuke idu fluidno. Potrudi se da uvod u opis bude interesantan. Trenutni film je {movie}\nOpis: {desc}\n. Napravi opis izmedju 40 i 70 reci. Recenicu obavezno zavrsi smisleno.Ukoliko je neophodno da zavrsis recenicu, pregazi ogranicenje broja reci, ali bez obzira na sve zavrsi recenicu do kraja!Komuniciraj na {self.language_}."
            }

            MAX_HISTORY = 3

            response = self.client_.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages + [movieMessage],
            max_tokens=160,
            temperature=0.7
            )
            print(f"Movie: ### {movie} ###")
            print(response.choices[0].message.content)
            print("---------------------------------------")
            movieNum += 1

            reply = response.choices[0].message.content

            messages.append(movieMessage)
            messages.append({
            "role": "assistant",
            "content": reply
            })

            messages = messages[-MAX_HISTORY:]

        return 1

In [ ]:
# Local emb
import numpy as np
from collections import Counter
import json
import os
import pickle
import random
import spacy


# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

class local_search_agent:
    def __init__(self, input_filepath=""):
        self.embeddings = np.load("embeddings.pkl", allow_pickle=True)
        self.movie_IDF = np.load("movie_IDF.pkl", allow_pickle=True)
        self.K = 3
    def preprocess_raw_data(self, input_filepath, output_json):

        """
        :param input_filepath: filepath of file with format:
        output_json: .json file containing dict:
            key = Movie_title
            value = preprocessed movie_description

        :return: dict that was put in .json file
        """
        processed_corpus = []
        titles = []
        with open(input_filepath, 'r', encoding='utf-8') as file:
            i = 0
            for line in file:
                # making sure that the preprocessing is happening
                i += 1
                if i % 100 == 0:
                    print(i)
                # skip empty lines
                if not line.strip():
                    continue

                # split the line by the first '|' character
                parts = line.split('|', 1)

                # extract the text description right side of '|'
                # if a line doesn't have a '|', process the whole line
                if len(parts) == 2:
                    raw_text = parts[1]
                    titles.append(parts[0])
                else:
                    raw_text = parts[0]

                # process the text through spaCy
                doc = nlp(raw_text)

                # list comprehension to clean and lemmatize the tokens
                words = [
                    token.lemma_.lower()  # convert to base form and lowercase
                    for token in doc
                    if token.is_alpha  # keeps only alphabetic characters
                ]

                # if the description wasn't empty after cleaning, add it to our corpus
                if words:
                    processed_corpus.append(words)

        output_data = dict(zip(titles, processed_corpus)) #create title:desc_list dict
        # save in json file
        with open(output_json, 'w', encoding='utf-8') as f:
            json.dump(output_data, f)

        return output_data
    def preprocessing_2(self, preprocessed_movie_data, min_count, emb_dim):
        """
        Further preprocessing of data that we got in preprocess_raw_data

        :param preprocessed_movie_data: output of  preprocess_raw_data function,
                                can be string - .json filename,
                                else it is dict, direct output of that function
        :param min_count: minimum total occurrences of a word in corpus, if a word
                    occurs less than min_count times we discard it
        :param emb_dim: dimension (length) of embedding vector

        :return: text_metadata, containing everyhing we need for word2vec algorithm
        """
        # extract titles and descriptions
        if isinstance(preprocessed_movie_data, str):
            with open(preprocessed_movie_data, "r", encoding="utf-8") as f:
                text_data_dict = json.load(f)
                text_data = list(text_data_dict.values())
                titles = list(text_data_dict.keys())
        else:
            text_data_dict = preprocessed_movie_data
            text_data = list(text_data_dict.values())
            titles = list(text_data_dict.keys())

        cleaned_data = []

        words = [word for movie_words in text_data for word in movie_words]
        word_freqs = Counter(words)

        # discarding all the words that appear less than min_count times in text
        words = [word for word in words if word_freqs[word] >= min_count]
        N = len(words) # total number of words in descriptions
        word_freqs = Counter({word: cnt for word, cnt in word_freqs.items() if cnt >= min_count})
        words_unique = list(word_freqs.keys())
        V = len(words_unique)  # number of different words in descriptions

        # removing rare words from movie descriptions
        for movie_list in text_data:
            filtered_movie = [word for word in movie_list if word_freqs[word] >= min_count]
            if filtered_movie:
                cleaned_data.append(filtered_movie)

        freqs = np.array(list(word_freqs.values()), dtype="float")
        freqs /= freqs.sum()  # normalize, get actual frequencies(probabilities)

        t = 1e-5
        central_word_prob = np.minimum(1.0, np.sqrt(t / freqs) + t / freqs)
        central_word_prob = dict(zip(words_unique, central_word_prob))
        # probability of not discarding a specific occurrence of word as central

        neg_sampling_word_prob = freqs ** 0.75
        neg_sampling_word_prob /= neg_sampling_word_prob.sum()  # again, normalize
        neg_sampling_word_prob = list(neg_sampling_word_prob)  # distribution needed for negative sampling

        word_to_idx = {w: i for i, w in enumerate(words_unique)}
        titles_to_idx = dict(zip(titles, word_to_idx.values()))
        # needed for faster reaching of rows and columns of weight matrices

        d = emb_dim  # number of nodes in hidden layer (the number of dimensions in a space of vectors the words live in)
        K = 5  # number of words for negative sampling
        m = 3  # window radius

        print('Done.')

        # creating text_metadata
        text_metadata = {
            "cleaned_data": cleaned_data,
            "words_unique": words_unique,
            "word_to_idx": word_to_idx,
            "titles_to_idx": titles_to_idx,
            "central_word_prob": central_word_prob,
            "neg_sampling_word_prob": neg_sampling_word_prob,
            "neg_samp": K,
            "window_radius": m,
            "total_words_num": N,
            "unique_words_num": V,
            "emb_dim": d
        }
        # saving text_metadata in .pkl file
        with open("text_metadata.pkl", "wb") as f:
            pickle.dump(text_metadata, f)

        print("Metadata saved.")
        return text_metadata
    def weight_init(self, arg):
        """
        Initializes or loads weight matrices.

        Parameters:
            arg: tuple (V, d) OR string (path to .pkl file)

        Returns:
            W_in, W_out
        """

        # case 1: tuple (V, d) - random initial weights
        if isinstance(arg, tuple):
            if len(arg) != 2:
                raise ValueError("Tuple must be of length 2: (V, d)")
            V, d = arg
            rng = np.random.default_rng()
            W_in = rng.uniform(-0.5 / d, 0.5 / d, (V, d))
            W_out = rng.uniform(-0.5 / d, 0.5 / d, (d, V))

            return W_in, W_out

        # case 2: string (path to .pkl file) - imported weights
        elif isinstance(arg, str):
            if not arg.endswith(".pkl"):
                raise ValueError("File must be a .pkl file")
            if not os.path.exists(arg):
                raise FileNotFoundError(f"{arg} does not exist")

            with open(arg, "rb") as f:
                data = pickle.load(f)
                if "W_in" not in data or "W_out" not in data:
                    raise KeyError("Dictionary must contain 'W_in' and 'W_out'")
                W_in = data["W_in"]
                W_out = data["W_out"]

                return W_in, W_out

        return 0

    # helper functions for word2vec
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def get_window(self, words, cent, radius):
        low = max(cent - radius, 0)
        high = min(cent + radius + 1, len(words) - 1)
        window = [word for word in words[low:high] if word != words[cent]]
        # remove occurrences of the exact word w to get a final window

        return window
    def negative_sampling(self, word_in, word_out, word_bank, weights, K):
        neg_sampled_words = random.choices(word_bank, weights=weights, k=K)
        while (word_in in neg_sampled_words) or (word_out in neg_sampled_words):
            neg_sampled_words = random.choices(word_bank, weights=weights, k=K)

        return neg_sampled_words
    def get_indexes(self, word_to_idx, word_in, word_out, neg_sampled_words):
        in_idx = word_to_idx[word_in]
        out_idx = word_to_idx[word_out]
        neg_samp_idxs = np.array([word_to_idx[w] for w in neg_sampled_words], dtype=int)

        return in_idx, out_idx, neg_samp_idxs
    def monitoring(self, i, W_in, Loss):
        print('iteration: ', i)
        print('Average loss: ', Loss)
        max_norm = np.max(np.linalg.norm(W_in, axis=1))
        print('Max embedding norm: ', max_norm)
        print('________________________________')

    def word2vec(self, max_iters, epochs, start_lr, monitoring_period, W_in_init, W_out_init, metadata):
        """
        Main loop for word2vec algorithm

        :param max_iters: maximum number of iterations
        :param epochs: number of epochs
        :param start_lr: start learning rate
        :param monitoring_period: period for monitoring (print current progress)
        :param W_in_init: input embeddings
        :param W_out_init: output embeddings
        :param metadata: text_metadata
        :return: final W_in, W_out
        """
        rng = np.random.default_rng()
        Loss = 0  # Loss that will be used for monitoring
        W_in = W_in_init
        W_out = W_out_init
        cleaned_data = metadata["cleaned_data"]
        words_unique = metadata["words_unique"]
        word_to_idx = metadata["word_to_idx"]
        central_word_prob = metadata["central_word_prob"]
        neg_sampling_word_prob = metadata["neg_sampling_word_prob"]
        K = metadata["neg_samp"]
        m = metadata["window_radius"]
        for ep in range(epochs):
            i_glob = -1
            used = 0  # for averaging Loss
            for movie in cleaned_data:
                i = -1
                for w in movie:
                    i_glob += 1
                    i += 1
                    if i_glob >= max_iters:
                        break
                    if (i_glob > 0) and (
                            i_glob % monitoring_period == 0):  # Monitoring done once in every monitoring_period central words
                        Loss /= used
                        self.monitoring(i_glob, W_in, Loss)
                        Loss = 0
                        used = 0
                    cont_flag = rng.binomial(n=1, p=central_word_prob[w])  # Skipping central word with some probability
                    if cont_flag == 0:
                        continue

                    lr = start_lr * (1 - (i_glob + ep * max_iters) / (epochs * max_iters))
                    window = self.get_window(movie, i, m)
                    for wo in window:

                        # negative sampling
                        neg_sampled_words = self.negative_sampling(w, wo, words_unique, neg_sampling_word_prob, K)

                        # determining rows and columns of matrices that are going to be changed
                        in_idx, out_idx, neg_samp_idxs = self.get_indexes(word_to_idx, w, wo, neg_sampled_words)

                        # W_in size is (V,d)
                        # W_out size is (d,V)
                        # size of the following vectors is (d,)
                        v_i = W_in[in_idx, :]
                        u_o = W_out[:, out_idx]
                        u_nk = W_out[:, neg_samp_idxs]

                        Loss += np.logaddexp(0, -np.dot(u_o, v_i)) + np.sum(np.logaddexp(0, np.dot(u_nk.T, v_i)))
                        # accumulating Loss for monitoring

                        dL_dvi = -(1 - self.sigmoid(np.dot(u_o, v_i))) * u_o  # gradient component for input
                        for j in range(K):
                            dL_dvi += self.sigmoid(np.dot(u_nk[:, j], v_i)) * u_nk[:, j]
                        W_in[in_idx, :] -= lr * dL_dvi.T

                        dL_duo = -(1 - self.sigmoid(np.dot(u_o, v_i))) * v_i  # gradient component for output
                        W_out[:, out_idx] -= lr * dL_duo

                        for j in range(K):
                            dL_dunk = self.sigmoid(
                                np.dot(u_nk[:, j], v_i)) * v_i  # gradient components for negative sampled words
                            W_out[:, neg_samp_idxs[j]] -= lr * dL_dunk
                        # weight updates
                    used += len(window)

            print('Epoch ', ep + 1, ' done.')

        return W_in, W_out
    def save_embeddings(self, W_in, W_out, word_to_idx, filename):
        """
        Saving the embeddings
        :param W_in: input embeddings
        :param W_out: output embeddings
        :param word_to_idx: dict that connects words to their
                            embedding vectors (columns of embedding matrices)
        :param filename: name of .pkl file in which the embeddings
                        will be saved
        """
        data = {
            "W_in": W_in,
            "W_out": W_out,
            "word_to_idx": word_to_idx
        }
        with open("embeddings.pkl", "wb") as f:
            pickle.dump(data, f)

        return 0
    def movie_IDF(self, text_metadata, embeddings):
        """
        Creating movie description embeddings along with some metadata
        :param text_metadata: text_metadata
        :param embeddings: word embedding matrix
        """
        embs = embeddings['W_out']
        data = text_metadata['cleaned_data']
        word_to_idx = text_metadata['word_to_idx']
        words = text_metadata['words_unique']
        V = text_metadata['unique_words_num']
        M = len(text_metadata['cleaned_data'])  # number of movies
        d = text_metadata['emb_dim']

        term_freqs = np.zeros((V, M))
        inverse_doc_freq = np.zeros((V, 1))
        movie_vecs = np.zeros((d, M))

        for w in text_metadata['words_unique']:
            for i in range(M):
                term_freqs[word_to_idx[w], i] = data[i].count(w)
                if term_freqs[word_to_idx[w], i] > 0:
                    inverse_doc_freq[word_to_idx[w]] += 1

        inverse_doc_freq = 1 / inverse_doc_freq
        inverse_doc_freq *= M
        inverse_doc_freq = np.log(inverse_doc_freq)
        term_freqs = term_freqs * inverse_doc_freq
        for i in range(M):
            for w in data[i]:
                movie_vecs[:, i] += term_freqs[word_to_idx[w], i] * embs[:, word_to_idx[w]]
            movie_vecs[:, i] /= np.sum(term_freqs[:, i])
        movie_vecs = movie_vecs / np.linalg.norm(movie_vecs, axis=0, keepdims=True)
        movieIDF = {
            "movie_vecs": movie_vecs,
            "IDF": inverse_doc_freq,
            "word_to_idx": word_to_idx,
            "titles_to_idx": text_metadata['titles_to_idx']
        }
        with open("movie_IDF.pkl", "wb") as f:
            pickle.dump(movieIDF, f)

    def topK_movies(self, description):
        """
        Providing top K choices of movies for given description
        :param description: description given by user
        :param movie_IDF_f: movie vectors with some metadata
        :param embeddings_f: word embeddings
        :param K: nubmer of movies to recommend
        :return: dict titles: similarity_to_description_scores
        """
        desc = nlp(description)
        # List comprehension to clean and lemmatize the tokens
        words = [
            token.lemma_.lower()  # Convert to base form and lowercase
            for token in desc
            if token.is_alpha  # Keeps ONLY alphabetic characters (removes punctuation, numbers, and symbols)
        ]

        embs = self.embeddings['W_out']
        word_to_idx = self.movie_IDF['word_to_idx']
        d = embs.shape[0]
        desc_vector = np.zeros((d, 1))
        IDF = self.movie_IDF['IDF']

        sum_IDF = 0
        for w in words:
            if w not in word_to_idx.keys():
                continue
            sum_IDF += IDF[word_to_idx[w]]
            desc_vector += IDF[word_to_idx[w]] * embs[:, word_to_idx[w]].reshape(-1, 1)
            desc_vector /= sum_IDF
            #desc_vectorAAAAA = desc_vector : put it in return if needed
            desc_vector /= np.linalg.norm(desc_vector, axis=0, keepdims=True)
        movie_vecs = self.movie_IDF['movie_vecs']
        embs_norm = embs / np.linalg.norm(embs, axis=0, keepdims=True)

        scores = (movie_vecs.T @ desc_vector).flatten()
        top_k_idx = np.argsort(scores)[-self.K:][::-1]
        top_k_values = scores[top_k_idx]
        titles_to_idx = self.movie_IDF['titles_to_idx']

        inv_map = {v: k for k, v in titles_to_idx.items()}
        keys = [inv_map[i] for i in top_k_idx]

        return dict(zip(keys, top_k_values))
    def start_interaction(self):
        print("Write the keywords you want to watch: ")
        text = input()

        max_rejection = 1
        rejection = 0
        while True:
            if rejection >= max_rejection:
                break
            print("Give more keywords: ")
            tmp = input()
            text += tmp

            rejection += 1

        return text
    def final_answer(self, movies):
        k = 1
        for movie in movies:
            print(f"Recommendation number {k}: {movie}")
            k += 1

In [ ]:
import csv

class SearchSystem:
    def __init__(self, sa: SearchAgent,saL: local_search_agent ,gnn: GNN, fcnn: FCNN):
        self.sa_ = sa
        self.saL_ = saL
        self.gnn_ = gnn
        self.fcnn_ = fcnn

        self.language_ = "Engleski"

        self.dataAdmin_ = pd.read_csv('adminsCredentials.csv')
        self.dataUser_ = pd.read_csv('usersCredentials.csv')
        self.guestNum_ = 10

        self.gnnActive_ = True
        self.fcnnActive_ = False
        self.myEmbActive_ = False
        self.oaiEmbActive_ = True


        self.loadAdminStatus()
    def LogIn(self, uType):
        if uType == "a":
            data = self.dataAdmin_
        if uType == "u":
            data = self.dataUser_
        users = dict(zip(data['userName'], data['password']))

        while True:
            userName = input("User name: ")
            password = input("Pasword: ")
            if users.get(userName) == password:
                break
        return userName ,True
    def updateAdminStatus(self):
        with open("adminOptions.csv", "w") as f:
            f.write("option,status\n")
            f.write(f"gnnActive,{self.gnnActive_}\n")
            f.write(f"fcnnActive,{self.fcnnActive_}\n")
            f.write(f"myEmbActive,{self.myEmbActive_}\n")
            f.write(f"oaiEmbActive,{self.oaiEmbActive_}\n")
    def loadAdminStatus(self):
        with open("adminOptions.csv", "r") as f:
            reader = csv.DictReader(f)
            k = 0
            for row in reader:
                value = row["status"].lower() == "true"

                if k == 0:
                    self.gnnActive_ = value
                if k == 1:
                    self.fcnnActive_ = value
                if k == 2:
                    self.myEmbActive_ = value
                if k == 3:
                    self.oaiEmbActive_ = value

                k += 1
    def prepareGuest(self):
        print("Kako se zovete: ")
        name = input()
        print(f"Kako bi dobio bolju preporuku {name}, sada ćeš da oceniš naše najpopularnije filmove.")
        print("Ukoliko neke od filmova nisi gledao, bilo bi dobro da po nekom znaju, iskustvu ili osećaju")
        print("dodeliš ocenu. Ocene su brojevi u intervalu 0 - 5")

        k = 0

        prepRatings = []
        for movie in self.gnn_.Data_.gList_:
            if k == self.guestNum_:
                break
            print(f"{movie}: ")
            rat = int(input())
            prepRatings.append(rat)
            k += 1

        return name, prepRatings
    def betterGuestExp(self):
        print("Do you want better exp. with our service as guest?")
        txt = input("(CHOSE AND TYPE): Yes - y, No - n: ")
        while txt != "y" and txt != "n":
            txt = input("y or n")

        if txt == "y":
            return True
        if txt == "n":
            return False
    def activeInteraction(self,uType,userName):
        if uType == "u":
            if self.oaiEmbActive_:
                self.sa_.clasicalInteraction(userName)
                resToken, inEmb = self.sa_.interactiveValidation()
                resDict = self.sa_.knnEmbd(inEmb)
                if self.gnnActive_:
                    Movies = self.gnn_.giveReccomendation(resDict, userName)
                if self.fcnnActive_:
                    Movies = self.fcnn_.giveRecommendation(resDict, userName)
                self.sa_.finalAnswer(userName, Movies)


            if self.myEmbActive_:
                resToken = self.saL_.start_interaction()
                resDict = self.saL_.topK_movies(resToken)
                if self.gnnActive_:
                    Movies = self.gnn_.giveReccomendation(resDict, userName)
                if self.fcnnActive_:
                    Movies = self.fcnn_.giveRecommendation(resDict, userName)
                self.saL_.final_answer(Movies)

        if uType == "g":
            be = self.betterGuestExp()
            if be:
                guestName, ratingSeed = self.prepareGuest()
            else:
                guestName = ""
                ratingSeed = None

            if self.oaiEmbActive_:
                self.sa_.clasicalInteraction(guestName)
                resToken, inEmb = self.sa_.interactiveValidation()
                resDict = self.sa_.knnEmbd(inEmb)
                if self.gnnActive_:
                    Movies = self.gnn_.giveReccomendation(resDict, guestName, ratingSeed)
                if self.fcnnActive_:
                    Movies = self.fcnn_.giveRecommendation(resDict, guestName, ratingSeed)
                self.sa_.finalAnswer(guestName, Movies)


            if self.myEmbActive_:
                resToken = self.saL_.start_interaction()
                resDict = self.saL_.topK_movies(resToken)
                if self.gnnActive_:
                    Movies = self.gnn_.giveReccomendation(resDict, guestName,ratingSeed)
                if self.fcnnActive_:
                    Movies = self.fcnn_.giveRecommendation(resDict, guestName,ratingSeed)
                self.saL_.final_answer(Movies)
    def workInTerminal(self):

        while True:
            print("Chose active model (fn - FCNN, gn - GNN)")
            txt = input('Type gn or fn: ')

            if txt == "gn":
                self.gnnActive_ = True
                self.fcnnActive_ = False
                break
            if txt == "fn":
                self.fcnnActive_ = True
                self.gnnActive_ = False
                break

        while True:
            print("Chose embeddings type ( m_emb - Our emb , oai_emb - OpenAi emb)")
            txt = input('Type m_emb or oai_emb: ')

            if txt == "m_emb":
                self.myEmbActive_ = True
                self.oaiEmbActive_ = False
                break
            if txt == "oai_emb":
                self.myEmbActive_ = False
                self.oaiEmbActive_ = True
                break
    def Active(self):
        print("#################################################")
        print("############ SEARCH ENGINE IS ACTIVE ############")
        print("#################################################")

        while True:
            print("######################")
            print("# Comands:")
            print("# STAY - use service #")
            print("# EXIT - end use     #")
            print("######################")

            comand = ""
            while True:
                comand = input("Comand: ")
                if comand == "EXIT" or comand == "STAY":
                    break

            if comand == "EXIT":
                break

            while True and comand == "STAY":
                print("CHOSE USER TYPE (g - guest, u - user, a - admin)")
                txt = input('Type g, u or a: ')

                adminActive = False
                guestActive = False
                userActive = False

                if txt == "a":
                    adminActive = True
                    break

                if txt == "g":
                    guestActive = True
                    break

                if txt == "u":
                    userActive = True
                    break

            ##############################
            ###### CHECK FOR GUEST  ######
            ##############################
            if  guestActive == True:
                self.activeInteraction("g","guest")

            ##############################
            ###### CHECK FOR GUEST  ######
            ##############################
            if userActive == True:
                userName, logState = self.LogIn("u")
                if logState:
                    self.activeInteraction("u",userName)

            ##############################
            ###### CHECK FOR ADMIN  ######
            ##############################
            if adminActive == True:
                _,logState = self.LogIn("a")
                if logState:
                    self.workInTerminal()
                    self.updateAdminStatus()

        print("COME AGAIN FOR BEST EXPERIENCE WITH MOVIE RECCOMENDATIONS")

In [ ]:
A = modelAndDataPreparation()

In [ ]:
GNN_models = GNN(A)

In [ ]:
SEARCH_agent = SearchAgent()

In [ ]:
SEARCH_agent_local = local_search_agent()

In [ ]:
#dict1 = SEARCH_agent_local.topK_movies("agression fight war drama love romance America")

In [ ]:
FCNN_models = FCNN(A)

In [ ]:
#FCNN_models.activeSpecificModel("'burbs, The (1989)","Maja",None)
#r = GNN_models.giveReccomendation(dict1,"Maja")

In [ ]:
SS = SearchSystem(SEARCH_agent, SEARCH_agent_local,GNN_models,FCNN_models)

In [17]:
SS.Active()

#################################################
############ SEARCH ENGINE IS ACTIVE ############
#################################################
######################
# Comands:
# STAY - use service #
# EXIT - end use     #
######################
COME AGAIN FOR BEST EXPERIENCE WITH MOVIE RECCOMENDATIONS
